# medianalysis — pipeline de extracción y análisis

```
retrieval/   → scraping + clasificación
factual/     → extracción, resolución global, grafo
rhetoric/    → sentiment, topics, framing, arguments
```

In [ ]:
import json
import pandas as pd

# retrieval
from medianalysis.retrieval.scraping import ScraperWorker
from medianalysis.retrieval.judges   import Ollama, SYSTEM_PROMPT, RESPONSE_SCHEMA

# factual
from medianalysis.factual.kb       import KBuilder, KB, SYSTEM_PROMPT as KB_PROMPT
from medianalysis.factual.grift    import EntityGrifter, RelationGrifter, ExplodeEventsWorker
from medianalysis.factual.embed    import build_embedding_input, MentionEmbedder, RelationEmbedder, EventEmbedder
from medianalysis.factual.cluster  import cluster_generic
from medianalysis.factual.canonize import build_el_input, build_canon_input, ELWorker, RelationCanonWorker, EventCanonWorker
from medianalysis.factual.backfill import backfill

# rhetoric
from medianalysis.rhetoric.sentiment import Sentimentalist
from medianalysis.rhetoric.topics    import fit_topics, assign_topics
from medianalysis.rhetoric.frame     import Framer
from medianalysis.rhetoric.argument  import ArgumentMiner, explode_arguments, link_arguments_to_kb

from medianalysis.distrib import merge_workers

## Parámetros globales

In [ ]:
TOTAL_WORKERS = 3
WID           = 0    # ← cambiar en cada Colab

---
# Retrieval

**Input:**  `urls.csv` — `id | url | media_name | publish_date | title`  
**Output:** `noticias_relevantes.csv` — artículos scrapeados y filtrados

In [ ]:
# Scraping
ScraperWorker(
    input_csv="urls.csv",
    output_csv="noticias_cuerpo.csv",
    id_col="id",
    batch_size=25,
    resume=True,
    total_workers=TOTAL_WORKERS,
    wid=WID,
).run()

merge_workers(output_csv="noticias_cuerpo.csv", id_col="id")

In [ ]:
# Clasificación de relevancia
Ollama(
    system_prompt=SYSTEM_PROMPT,
    response_schema=RESPONSE_SCHEMA,
    input_csv="noticias_cuerpo.csv",
    output_csv="clasificacion.csv",
    id_col="id",
    batch_size=25,
    resume=True,
    total_workers=TOTAL_WORKERS,
    wid=WID,
).run()

merge_workers(output_csv="clasificacion.csv", id_col="id")

In [ ]:
# Filtrar relevantes
cuerpo_df = pd.read_csv("noticias_cuerpo.csv")
clasif_df = pd.read_csv("clasificacion.csv")

relevantes = clasif_df[clasif_df["llm_class"] == "1"]["id"]
cuerpo_df[cuerpo_df["id"].isin(relevantes)].to_csv("noticias_relevantes.csv", index=False)

print(f"{len(relevantes)} relevantes de {len(cuerpo_df)} totales")

---
# Factual

## 1 — Extracción por documento

**Input:**  `noticias_relevantes.csv`  
**Output:** `extracciones.csv` — `id | entities | relations | events`

In [ ]:
KBuilder(
    system_prompt=KB_PROMPT,
    model_schema=KB,
    input_csv="noticias_relevantes.csv",
    output_csv="extracciones.csv",
    id_col="id",
    batch_size=25,
    resume=True,
    total_workers=TOTAL_WORKERS,
    wid=WID,
).run()

merge_workers(output_csv="extracciones.csv", id_col="id")

## 2 — Resolución global

### 2.1 — Explosión

In [ ]:
EntityGrifter(
    input_csv="extracciones.csv",
    output_csv="menciones_raw.csv",
    id_col="id", batch_size=100, resume=True,
).run()

RelationGrifter(
    input_csv="extracciones.csv",
    output_csv="relaciones_raw.csv",
    id_col="id", batch_size=100, resume=True,
).run()

ExplodeEventsWorker(
    input_csv="extracciones.csv",
    output_csv="eventos_raw.csv",
    id_col="id", batch_size=100, resume=True,
).run()

### 2.2 — Embeddings

In [ ]:
build_embedding_input(
    menciones_csv="menciones_raw.csv",
    extracciones_csv="extracciones.csv",
    cuerpo_csv="noticias_relevantes.csv",
    output_csv="menciones_embedding_input.csv",
)

MentionEmbedder(
    input_csv="menciones_embedding_input.csv",
    output_csv="menciones_embeddings.csv",
    id_col="mention_id", batch_size=500, resume=True,
).run()

RelationEmbedder(
    input_csv="relaciones_raw.csv",
    output_csv="relaciones_embeddings.csv",
    id_col="relation_id", batch_size=500, resume=True,
).run()

EventEmbedder(
    input_csv="eventos_raw.csv",
    output_csv="eventos_embeddings.csv",
    id_col="event_id", batch_size=500, resume=True,
).run()

### 2.3 — Clustering

In [ ]:
# Tabla de tipos para agrupar menciones
type_df = pd.DataFrame([
    {"mention_id": f"{doc['id']}__{ent['id']}", "type": ent["type"]}
    for _, doc in pd.read_csv("extracciones.csv").iterrows()
    for ent in json.loads(doc["entities"])
])

cluster_generic(
    embeddings_csv="menciones_embeddings.csv",
    id_col="mention_id", output_csv="clusters.csv", prefix="ent",
    min_cluster_size=2, group_by_col="type", group_source_df=type_df,
)

cluster_generic(
    embeddings_csv="relaciones_embeddings.csv",
    id_col="relation_id", output_csv="relaciones_clusters.csv",
    prefix="rel", min_cluster_size=3,
)

cluster_generic(
    embeddings_csv="eventos_embeddings.csv",
    id_col="event_id", output_csv="eventos_clusters.csv",
    prefix="evt", min_cluster_size=2,
)

### 2.4 — Entity Linking

In [ ]:
build_el_input(
    clusters_csv="clusters.csv",
    extracciones_csv="extracciones.csv",
    cuerpo_csv="noticias_relevantes.csv",
    output_csv="clusters_para_el.csv",
)

ELWorker(
    input_csv="clusters_para_el.csv",
    output_csv="entidades_canonicas.csv",
    id_col="cluster_id", batch_size=10, resume=True,
).run()

### 2.5 — Canonicalización de relaciones y eventos

In [ ]:
build_canon_input(
    clusters_csv="relaciones_clusters.csv", raw_csv="relaciones_raw.csv",
    id_col="relation_id", text_cols=["relation", "evidence"],
    output_csv="relaciones_para_canon.csv",
)
RelationCanonWorker(
    input_csv="relaciones_para_canon.csv", output_csv="tipos_relacion.csv",
    id_col="cluster_id", batch_size=10, resume=True,
).run()

build_canon_input(
    clusters_csv="eventos_clusters.csv", raw_csv="eventos_raw.csv",
    id_col="event_id", text_cols=["event_type", "trigger"],
    output_csv="eventos_para_canon.csv",
)
EventCanonWorker(
    input_csv="eventos_para_canon.csv", output_csv="tipos_evento.csv",
    id_col="cluster_id", batch_size=10, resume=True,
).run()

### 2.6 — Backfill → grafo

In [ ]:
backfill(
    extracciones_csv="extracciones.csv",
    menciones_raw_csv="menciones_raw.csv",
    clusters_csv="clusters.csv",
    entidades_csv="entidades_canonicas.csv",
    relaciones_raw_csv="relaciones_raw.csv",
    relaciones_clusters_csv="relaciones_clusters.csv",
    tipos_relacion_csv="tipos_relacion.csv",
    eventos_raw_csv="eventos_raw.csv",
    eventos_clusters_csv="eventos_clusters.csv",
    tipos_evento_csv="tipos_evento.csv",
    output_grafo_csv="grafo.csv",
    output_eventos_csv="eventos_resueltos.csv",
)

---
# Rhetoric

Los cuatro componentes son independientes entre sí.

In [ ]:
# Sentiment
Sentimentalist(
    input_csv="noticias_relevantes.csv",
    output_csv="sentiment.csv",
    id_col="id", batch_size=50, resume=True,
).run()

In [ ]:
# Topics
topic_model = fit_topics(
    cuerpo_csv="noticias_relevantes.csv",
    output_docs_csv="topics_por_doc.csv",
    output_topics_csv="topics_keywords.csv",
)

In [ ]:
# Framing
Framer(
    input_csv="noticias_relevantes.csv",
    output_csv="framing.csv",
    id_col="id", batch_size=25, resume=True,
    total_workers=TOTAL_WORKERS, wid=WID,
).run()

merge_workers(output_csv="framing.csv", id_col="id")

In [ ]:
# Arguments
ArgumentMiner(
    input_csv="noticias_relevantes.csv",
    output_csv="arguments_raw.csv",
    id_col="id", batch_size=25, resume=True,
    total_workers=TOTAL_WORKERS, wid=WID,
).run()

merge_workers(output_csv="arguments_raw.csv", id_col="id")

explode_arguments(raw_csv="arguments_raw.csv", output_csv="arguments.csv")

link_arguments_to_kb(
    arguments_csv="arguments.csv",
    extracciones_csv="extracciones.csv",
    menciones_raw_csv="menciones_raw.csv",
    clusters_csv="clusters.csv",
    entidades_csv="entidades_canonicas.csv",
    output_csv="arguments_linked.csv",
)